# 01 · Exploring the Bengaluru listings

Quick look at the cleaned data (`data/processed/clean.csv`, produced by `python src/clean.py`). Four questions: how are prices spread, how much does area explain, how much does locality matter, and what does BHK add on top of area?

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

BLUE = '#2a78d6'          # single-series colour
pio.templates.default = 'plotly_white'

def fmt_price(lakhs):
    """Show lakhs below 1 Cr and crores above (100 lakhs = 1 crore)."""
    return f'₹{lakhs / 100:g} Cr' if lakhs >= 100 else f'₹{lakhs:g}L'

PRICE_TICKS = [10, 20, 50, 100, 200, 500, 1000, 2000]
PRICE_TEXT = [fmt_price(v) for v in PRICE_TICKS]

def style(fig, **kw):
    fig.update_layout(font=dict(size=13, color='#0b0b0b'), title_font_size=16,
                      margin=dict(l=60, r=20, t=60, b=50), **{'height': 420, **kw})
    fig.update_xaxes(gridcolor='#ecebe8', zeroline=False)
    fig.update_yaxes(gridcolor='#ecebe8', zeroline=False)
    return fig

df = pd.read_csv('../data/processed/clean.csv')
df['price_per_sqft'] = df.price_lakhs * 1e5 / df.area_sqft   # for exploration only, never a model input
print(f'{len(df):,} listings, {df.location.nunique():,} localities')
df.describe().round(1)

11,740 listings, 1,181 localities


,ready_to_move,area_sqft,bhk,bathrooms,balconies,price_lakhs,price_per_sqft
count,11740.0,11740.0,11740.0,11740.0,11740.0,11740.0,11740.0
mean,0.8,1564.6,2.6,2.6,1.6,105.4,6144.6
std,0.4,951.1,0.9,1.0,0.8,116.5,3010.6
min,0.0,300.0,1.0,1.0,0.0,10.0,2500.0
25%,1.0,1115.0,2.0,2.0,1.0,50.0,4257.2
50%,1.0,1300.0,3.0,2.0,2.0,70.0,5319.1
75%,1.0,1700.0,3.0,3.0,2.0,115.0,6910.9
max,1.0,30400.0,10.0,12.0,3.0,2200.0,21739.1


## 1. Price distribution

Prices are heavily right-skewed. On a log scale they look roughly bell-shaped, which is why the model will be trained on `log(price)`.

In [2]:
# Bin log10(price) ourselves so the bars are evenly spaced on a log scale
counts, edges = np.histogram(np.log10(df.price_lakhs), bins=50)
lo, hi = 10 ** edges[:-1], 10 ** edges[1:]
fig = go.Figure(go.Bar(
    x=(edges[:-1] + edges[1:]) / 2, y=counts, width=np.diff(edges) * 0.9, marker_color=BLUE,
    customdata=np.c_[lo, hi], hovertemplate='₹%{customdata[0]:.0f}L to ₹%{customdata[1]:.0f}L<br>%{y} listings<extra></extra>'))
fig.update_xaxes(tickvals=np.log10(PRICE_TICKS), ticktext=PRICE_TEXT, title='Price (log scale)')
med = df.price_lakhs.median()
fig.add_vline(np.log10(med), line_dash='dash', line_color='#52514e', annotation_text=f'median {fmt_price(med)}')
fig.update_layout(title='Most listings sit between ₹40L and ₹1.5 Cr, with a long expensive tail')
style(fig, yaxis_title='Listings').show()
print('skew raw:', round(df.price_lakhs.skew(), 2), '| skew log:', round(np.log(df.price_lakhs).skew(), 2))

skew raw: 6.09 | skew log: 0.8


## 2. Price vs area

Area is the single strongest driver. The vertical spread at any given area is what locality and other features have to explain.

In [3]:
s = df.sample(4000, random_state=0)
fig = px.scatter(s, x='area_sqft', y='price_lakhs', log_x=True, log_y=True, opacity=0.35,
                 color_discrete_sequence=[BLUE], hover_data=['location', 'bhk'],
                 labels={'area_sqft': 'Area (sq ft, log)', 'price_lakhs': 'Price (₹ lakhs, log)'},
                 title='Bigger homes cost more, but the same area can differ 3x in price')
fig.update_traces(marker_size=6)
fig.update_yaxes(tickvals=PRICE_TICKS, ticktext=PRICE_TEXT, title='Price (log)')
fig.update_xaxes(tickvals=[300, 500, 1000, 2000, 5000, 10000, 20000])
style(fig).show()
print('correlation of log(area) and log(price):', round(np.corrcoef(np.log(df.area_sqft), np.log(df.price_lakhs))[0, 1], 3))

correlation of log(area) and log(price): 0.839


## 3. Locality matters a lot

Price per sq ft for the 15 localities with the most listings. Same city, very different rates; this is why locality is a key feature.

In [4]:
top = df.location.value_counts().head(15).index
t = df[df.location.isin(top)]
order = t.groupby('location').price_per_sqft.median().sort_values().index.tolist()
fig = px.box(t, y='location', x='price_per_sqft', orientation='h', points='outliers',
             category_orders={'location': order}, color_discrete_sequence=[BLUE],
             labels={'price_per_sqft': 'Price per sq ft (₹)', 'location': ''},
             title='Typical rate ranges from about ₹3,000 to ₹7,200 per sq ft across busy localities')
fig.update_traces(marker=dict(size=3, opacity=0.4))
style(fig, height=520).show()
t.groupby('location').price_per_sqft.agg(['count', 'median']).round(0).sort_values('median')

,count,median
location,,
Electronic City Phase II,120,2976.0
Uttarahalli,151,4087.0
Raja Rajeshwari Nagar,156,4235.0
Electronic City,269,4667.0
Kanakpura Road,238,5025.0
Yelahanka,198,5056.0
Bannerghatta Road,147,5329.0
Marathahalli,164,5500.0
Whitefield,515,5618.0


## 4. BHK on top of area

More bedrooms means a higher price, mostly because the home is bigger. Price per sq ft is flat from 1 to 3 BHK, then jumps for 4+ BHK; many of those are independent houses on plots (plots median about ₹8,700/sq ft vs about ₹5,200 for apartments), so `area_type` is a useful feature.

In [5]:
d = df[df.bhk <= 6]
fig = px.box(d, x='bhk', y='price_lakhs', log_y=True, points=False, color_discrete_sequence=[BLUE],
             labels={'bhk': 'BHK', 'price_lakhs': 'Price (₹ lakhs, log)'},
             title='Price rises with BHK, with a big jump at 4 BHK')
fig.update_yaxes(tickvals=PRICE_TICKS, ticktext=PRICE_TEXT, title='Price (log)')
style(fig).show()
d.groupby('bhk').agg(listings=('price_lakhs', 'size'), median_price=('price_lakhs', 'median'),
                     median_ppsf=('price_per_sqft', 'median')).round(0)

,listings,median_price,median_ppsf
bhk,,,
1,589,35.0,5190.0
2,5164,55.0,4844.0
3,4501,90.0,5587.0
4,1145,220.0,8174.0
5,179,240.0,8000.0
6,82,200.0,6661.0


## Other features at a glance

In [6]:
print(df.groupby('area_type').price_per_sqft.agg(['count', 'median']).round(0))
print()
print(df.groupby('ready_to_move').price_per_sqft.agg(['count', 'median']).round(0))
vc = df.location.value_counts()
print(f'\nLocalities with >= 10 listings: {(vc >= 10).sum()} covering {vc[vc >= 10].sum() / len(df):.0%} of rows')

                count  median
area_type                    
Built-up         2285  5093.0
Carpet             82  5903.0
Plot             1238  8667.0
Super built-up   8135  5167.0

               count  median
ready_to_move               
0               2495  5302.0
1               9245  5325.0

Localities with >= 10 listings: 226 covering 78% of rows
